# 14 - Multi-seed (local-z) + FedProx same-config (Paper 4, review-6 P1+P5)

**Dua perbaikan reviewer dalam satu notebook:**

**P1 (statistik local-z):** uji 5-seed cross-dataset FedAvg vs local-only, tetapi
GLOBAL dievaluasi dgn **deployment-consistent local normalization** (tiap domain
pakai z-score train domainnya), bukan pooled-z. Hitung mean+/-std, 95% CI,
paired t-test, Wilcoxon, Cohen's d. Ini mengganti statistik lama (pooled-z).

**P5 (FedProx same-config):** ulangi FedAvg vs FedProx(mu) memakai **arsitektur
yang SAMA dgn main model** (9->128->64->1), evaluasi local-z, agar sebanding dgn
Tabel cross-dataset (bukan arsitektur kecil seperti nb06).

**Konsistensi:** MLP 9->128->64->1, ROUNDS=100, LOCAL_EPOCHS=5, LR=3e-3.
Output: `fed_out/multiseed_localz.csv`, `fedprox_samearch.csv`.

> Jalankan setelah nb01. JANGAN mengarang angka.

## 0. Setup + MLP (identik main model) + FedProx proximal

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','pandas','scipy','scikit-learn'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, numpy as np, pandas as pd
from sklearn.metrics import matthews_corrcoef
from scipy import stats
OUTDIR='fed_out'; os.makedirs(OUTDIR, exist_ok=True)
ROUNDS=100; LOCAL_EPOCHS=5; LR=3e-3; LAYERS=[9,128,64,1]
SEEDS=[13,42,101,202,303]
def _relu(x): return np.maximum(0,x)
def _sig(x): return 1.0/(1.0+np.exp(-np.clip(x,-30,30)))
def init_w(seed):
    r=np.random.RandomState(seed); W=[]
    for a,b in zip(LAYERS[:-1],LAYERS[1:]):
        W.append(r.randn(a,b)*np.sqrt(2.0/a)); W.append(np.zeros(b))
    return W
def forward(W,X):
    a1=X@W[0]+W[1]; z1=_relu(a1); a2=z1@W[2]+W[3]; z2=_relu(a2)
    a3=z2@W[4]+W[5]; return _sig(a3).ravel(),(X,a1,z1,a2,z2)
def train_local(W,X,y,epochs=LOCAL_EPOCHS,lr=LR,batch=256,mu=0.0,Wg=None,seed=0):
    r=np.random.RandomState(seed); n=len(y); W=[w.copy() for w in W]
    for _ in range(epochs):
        idx=r.permutation(n)
        for s in range(0,n,batch):
            bi=idx[s:s+batch]; Xb=X[bi]; yb=y[bi].astype(float)
            out,(X_,a1,z1,a2,z2)=forward(W,Xb); g=(out-yb)/len(yb)
            gW4=z2.T@g[:,None]; gb4=g.sum(0)
            d2=(g[:,None]@W[4].T)*(a2>0); gW2=z1.T@d2; gb2=d2.sum(0)
            d1=(d2@W[2].T)*(a1>0); gW0=X_.T@d1; gb0=d1.sum(0)
            gall=[gW0,gb0,gW2,gb2,gW4,gb4]
            for i,gv in enumerate(gall):
                gv=np.asarray(gv).reshape(W[i].shape)
                if mu>0 and Wg is not None: gv=gv+mu*(W[i]-Wg[i])
                W[i]=W[i]-lr*gv
    return W
def fedavg(ws,sizes):
    tot=float(sum(sizes)); return [sum((sizes[k]/tot)*ws[k][i] for k in range(len(ws))) for i in range(len(ws[0]))]
def zfit(X): mu=X.mean(0); sd=X.std(0); sd[sd==0]=1.0; return mu,sd
def zap(X,mu,sd): return (X-mu)/sd
def pred(W,X,mu,sd): out,_=forward(W,zap(X,mu,sd)); return (out>=0.5).astype(int)
print('setup ok | LAYERS',LAYERS,'seeds',SEEDS)

## 1. Muat partisi

In [ ]:
def load(name):
    p=os.path.join(OUTDIR,name)
    if not os.path.exists(p): print('MISSING',p); return None,None
    d=np.load(p,allow_pickle=True); return np.asarray(d['X'],float), np.asarray(d['y'],int)
Xc_tr,yc_tr=load('cic_train.npz'); Xc_te,yc_te=load('cic_test.npz')
Xu_tr,yu_tr=load('unsw_train.npz'); Xu_te,yu_te=load('unsw_test.npz')
ok=all(v is not None for v in [Xc_tr,Xu_tr,Xc_te,Xu_te])
if ok:
    yg_te=np.concatenate([yc_te,yu_te])
    print('loaded | GLOBAL-test',len(yg_te))
else: print('(jalankan nb01 dulu)')

## 2. P1: 5-seed FedAvg (local-z) vs local-only + statistik

GLOBAL-test = CIC-test pakai z CIC-train; UNSW-test pakai z UNSW-train; gabung.
local-only = model terbaik antar-klien (konsisten definisi nb07).

In [ ]:
def mcc(y,yp): return matthews_corrcoef(y,yp)
fed_scores=[]; loc_scores=[]
if ok:
    zc_cic=zfit(Xc_tr); zc_un=zfit(Xu_tr)
    Xzc=zap(Xc_tr,*zc_cic); Xzu=zap(Xu_tr,*zc_un)
    sizes=[len(yc_tr),len(yu_tr)]
    for sd in SEEDS:
        # federated FedAvg
        Wg=init_w(sd)
        for t in range(ROUNDS):
            W1=train_local(Wg,Xzc,yc_tr,seed=sd); W2=train_local(Wg,Xzu,yu_tr,seed=sd)
            Wg=fedavg([W1,W2],sizes)
        # GLOBAL local-z: tiap domain z-score domainnya
        yp=np.concatenate([pred(Wg,Xc_te,*zc_cic),pred(Wg,Xu_te,*zc_un)])
        fed_scores.append(mcc(yg_te,yp))
        # local-only: latih tiap klien penuh, ambil MCC GLOBAL terbaik (murah hati)
        best=-1
        for (Xl,yl,zl) in [(Xc_tr,yc_tr,zc_cic),(Xu_tr,yu_tr,zc_un)]:
            Wl=init_w(sd); Wl=train_local(Wl,zap(Xl,*zl),yl,epochs=ROUNDS*LOCAL_EPOCHS,seed=sd)
            yl_g=np.concatenate([pred(Wl,Xc_te,*zc_cic),pred(Wl,Xu_te,*zc_un)])
            best=max(best,mcc(yg_te,yl_g))
        loc_scores.append(best)
        print(f'seed {sd}: fed(local-z)={fed_scores[-1]:.4f} local-only={loc_scores[-1]:.4f}')
    fed=np.array(fed_scores); loc=np.array(loc_scores)
    def ci95(a): m=a.mean(); h=1.96*a.std(ddof=1)/np.sqrt(len(a)); return m-h,m+h
    tt=stats.ttest_rel(fed,loc)
    try: ww=stats.wilcoxon(fed,loc)
    except Exception as e: ww=None; print('wilcoxon skip',e)
    diff=fed-loc; d=diff.mean()/ (diff.std(ddof=1) if diff.std(ddof=1)>0 else 1)
    rows=[{'setting':'federated-FedAvg(local-z)','mean':round(fed.mean(),4),'std':round(fed.std(ddof=1),4),
           'CI95_lo':round(ci95(fed)[0],4),'CI95_hi':round(ci95(fed)[1],4)},
          {'setting':'local-only(best)','mean':round(loc.mean(),4),'std':round(loc.std(ddof=1),4),
           'CI95_lo':round(ci95(loc)[0],4),'CI95_hi':round(ci95(loc)[1],4)}]
    sig={'mean_diff':round(float(diff.mean()),4),'paired_t_p':round(float(tt.pvalue),5),
         'wilcoxon_p':(round(float(ww.pvalue),5) if ww else None),'cohen_d':round(float(d),3),'n_seed':len(SEEDS)}
    pd.DataFrame(rows).to_csv(os.path.join(OUTDIR,'multiseed_localz.csv'),index=False)
    pd.DataFrame([sig]).to_csv(os.path.join(OUTDIR,'multiseed_localz_sig.csv'),index=False)
    import IPython.display as ipd; ipd.display(pd.DataFrame(rows)); ipd.display(pd.DataFrame([sig]))
    print('SIG(local-z):',sig)
print('=== SEL 2 (5-seed local-z) SELESAI ===')

## 3. P5: FedProx same-config (9->128->64->1), evaluasi local-z

mu in {0, 0.001, 0.01, 0.1}; arsitektur SAMA dgn main model (bukan kecil).
Evaluasi GLOBAL local-z, seed tetap 42 (sebanding Tabel cross-dataset).

In [ ]:
res_fp=[]
if ok:
    zc_cic=zfit(Xc_tr); zc_un=zfit(Xu_tr)
    Xzc=zap(Xc_tr,*zc_cic); Xzu=zap(Xu_tr,*zc_un); sizes=[len(yc_tr),len(yu_tr)]
    for mu in (0.0,0.001,0.01,0.1):
        Wg=init_w(42)
        for t in range(ROUNDS):
            W1=train_local(Wg,Xzc,yc_tr,mu=mu,Wg=Wg,seed=42)
            W2=train_local(Wg,Xzu,yu_tr,mu=mu,Wg=Wg,seed=42)
            Wg=fedavg([W1,W2],sizes)
        yp=np.concatenate([pred(Wg,Xc_te,*zc_cic),pred(Wg,Xu_te,*zc_un)])
        res_fp.append({'method':('FedAvg' if mu==0 else f'FedProx(mu={mu})'),'mu':mu,
                       'GLOBAL_localz_MCC':round(mcc(yg_te,yp),4)})
        print(res_fp[-1])
    pd.DataFrame(res_fp).to_csv(os.path.join(OUTDIR,'fedprox_samearch.csv'),index=False)
    import IPython.display as ipd; ipd.display(pd.DataFrame(res_fp))
print('=== SEL 3 (FedProx same-arch) SELESAI ===')

## 4. Unggah ke S3

In [ ]:
S3_BUCKET=os.environ.get('FED_S3_BUCKET','ssh-detection-features-232032302717')
REGION=os.environ.get('AWS_REGION','ap-southeast-1'); PREFIX='unsw-far/federated/results/multiseed_localz'
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for f in ('multiseed_localz.csv','multiseed_localz_sig.csv','fedprox_samearch.csv'):
        p=os.path.join(OUTDIR,f)
        if os.path.exists(p): s3.upload_file(p,S3_BUCKET,f'{PREFIX}/{f}'); up+=1; print('ok ->',f)
    print(f'unggah {up} -> s3://{S3_BUCKET}/{PREFIX}/')
except Exception as e: print('(upload gagal:',e,')')
print('=== SEL 4 (unggah) SELESAI ===')

## 5. Catatan untuk naskah

- `multiseed_localz*.csv` -> ganti statistik lama (pooled-z) di Sec cross-dataset:
  mean+/-std, CI95, paired-t p, Wilcoxon p, Cohen's d SEMUA pada local-z.
- `fedprox_samearch.csv` -> ganti Tabel FedProx (sekarang arsitektur SAMA 128-64,
  evaluasi local-z) -> hilangkan 'smaller-capacity config' caveat.
- Ganti placeholder [TODO] di naskah dgn angka NYATA dari sini.